# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields by @id
record_sets = []
print("Available Record Sets and Fields:")
for rs in metadata.record_sets:
    print(f"Record Set: {rs['@id']}, name: {rs.get('name')}")
    record_sets.append(rs['@id'])
    for field in rs.get('fields', []):
        print(f"  Field: {field['@id']}, name: {field.get('name')}, datatype: {field.get('dataType')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set (using @id references)
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only make DataFrame if records exist
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"  No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Filter, normalize, and group by field using @id references
# Please adjust the @id values and names based on overview above

if record_sets:
    sample_record_set = record_sets[0]
    df = dataframes.get(sample_record_set)
    if df is not None:
        print(f"Working on Record Set @id: {sample_record_set}")
        # Select a numeric field (edit this if different):
        numeric_fields = [col for col in df.columns if df[col].dtype in ('float64', 'int64')]
        if numeric_fields:
            numeric_field = numeric_fields[0]
            print(f"Using Numeric Field: {numeric_field}")

            threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold}:")
            display(filtered_df.head())

            # Normalize
            col_norm = f"{numeric_field}_normalized"
            filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, col_norm]].head())

            # Attempt grouping by another field if more than one exists
            group_fields = [col for col in df.columns if col != numeric_field]
            group_field = group_fields[0] if group_fields else None
            if group_field and group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
                display(grouped_df.head())
            else:
                print("No suitable group field found.")
        else:
            print("No numeric fields found in DataFrame.")
    else:
        print("No DataFrame loaded for the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets:
    sample_record_set = record_sets[0]
    df = dataframes.get(sample_record_set)
    if df is not None and numeric_fields:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field} in Record Set {sample_record_set}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
        
        # If grouping was possible, visualize group means
        if 'grouped_df' in locals():
            plt.figure(figsize=(10,5))
            sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
            plt.title(f"Average {numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(f"Mean {numeric_field}")
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we have loaded and explored the FAIR^2 mass spectrometry dataset using the Croissant metadata, retrieved field and record structure using `@id` references, performed basic data extraction, and applied fundamental EDA and visualizations. Further analysis can focus on deeper domain-specific questions and advanced modeling.